# Target Treasury Account Monitor Test

This notebook has two parts:

1. Offline smoke tests using simple mock IB-like objects.
2. Optional live IB calls. Keep `RUN_LIVE_IB = False` until IB Gateway/TWS is open and the account ID is filled.

In [14]:
from pathlib import Path
from types import SimpleNamespace
import sys

import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "target_treasury_account_monitor").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from target_treasury_account_monitor.config import DEFAULT_CLIENT_ID, DEFAULT_HOST, DEFAULT_PORT, MARKET_DATA_TYPES, MonitorSettings
from target_treasury_account_monitor.contracts import contract_label, is_treasury_contract
from target_treasury_account_monitor.frames import excluded_positions_frame, positions_to_frame
from target_treasury_account_monitor.greeks import greek_totals
from target_treasury_account_monitor.wechat import build_wechat_text

## Offline smoke test

This verifies treasury filtering, futures delta fallback, option Greeks aggregation, and excluded-position reporting without connecting to IB.

In [ ]:
future_contract = SimpleNamespace(
    conId=1,
    secType="FUT",
    symbol="ZF",
    localSymbol="ZFM6",
    tradingClass="ZF",
    exchange="CBOT",
    currency="USD",
    lastTradeDateOrContractMonth="202606",
    strike=0,
    right="",
    multiplier="1000",
)
fop_contract = SimpleNamespace(
    conId=2,
    secType="FOP",
    symbol="ZF",
    localSymbol="ZFM6 C 110",
    tradingClass="ZF",
    exchange="CBOT",
    currency="USD",
    lastTradeDateOrContractMonth="202606",
    strike=110,
    right="C",
    multiplier="1000",
)
stock_contract = SimpleNamespace(
    conId=3,
    secType="STK",
    symbol="AAPL",
    localSymbol="AAPL",
    tradingClass="AAPL",
    exchange="SMART",
    currency="USD",
    lastTradeDateOrContractMonth="",
    strike=0,
    right="",
    multiplier="1",
)

positions = [
    SimpleNamespace(account="TARGET", contract=future_contract, position=2, avgCost=108.5),
    SimpleNamespace(account="TARGET", contract=fop_contract, position=-3, avgCost=0.25),
]
all_positions = positions + [SimpleNamespace(account="TARGET", contract=stock_contract, position=10, avgCost=180)]

option_greeks = SimpleNamespace(impliedVol=0.12, delta=0.32, gamma=0.04, theta=-0.02, vega=0.11)
tickers = {
    1: SimpleNamespace(bid=108.25, ask=108.5, last=108.375, close=108, contract=future_contract),
    2: SimpleNamespace(bid=0.24, ask=0.26, modelGreeks=option_greeks, contract=fop_contract),
}
portfolio_map = {
    1: SimpleNamespace(marketPrice=108.375, marketValue=216750, unrealizedPNL=250, realizedPNL=0),
    2: SimpleNamespace(marketPrice=0.25, marketValue=-750, unrealizedPNL=-30, realizedPNL=0),
}

assert is_treasury_contract(future_contract)
assert is_treasury_contract(fop_contract)
assert not is_treasury_contract(stock_contract)
assert contract_label(future_contract) == "ZFM6"

frame = positions_to_frame(positions, tickers, portfolio_map)
totals = greek_totals(frame)
excluded = excluded_positions_frame(all_positions)

display(frame)
display(totals)
display(excluded)

In [ ]:
summary = pd.DataFrame(
    [
        {"account": "TARGET", "tag": "NetLiquidation", "value": 100000, "currency": "USD"},
        {"account": "TARGET", "tag": "ExcessLiquidity", "value": 80000, "currency": "USD"},
        {"account": "TARGET", "tag": "AvailableFunds", "value": 75000, "currency": "USD"},
    ]
)
settings = MonitorSettings(
    host=DEFAULT_HOST,
    port=DEFAULT_PORT,
    client_id=DEFAULT_CLIENT_ID,
    account="TARGET",
    market_data_type=MARKET_DATA_TYPES["Live"],
    quote_wait_seconds=2.0,
    refresh_seconds=5,
    auto_refresh=False,
    auto_reconnect=True,
    reconnect_backoff_seconds=10,
    wechat_webhook_url="",
    wechat_push_enabled=False,
    wechat_min_interval_seconds=300,
)
print(build_wechat_text(settings, summary, frame))

## Optional live IB test

Set `RUN_LIVE_IB = True` and fill `TARGET_ACCOUNT` only when you want the notebook to connect to IB Gateway/TWS.

In [15]:
RUN_LIVE_IB = True
TARGET_ACCOUNT = "U16251798"

if RUN_LIVE_IB:
    from ib_async import IB, util
    from ib_async.ib import StartupFetch
    from target_treasury_account_monitor.ib_client import account_summary_frame, fetch_target_positions, portfolio_items_by_key, subscribe_quotes_for_positions

    util.startLoop()
    ib = IB()
    ib.connect(DEFAULT_HOST, DEFAULT_PORT, clientId=DEFAULT_CLIENT_ID + 20, readonly=True, timeout=10, fetchFields=StartupFetch.POSITIONS | StartupFetch.ACCOUNT_UPDATES | StartupFetch.SUB_ACCOUNT_UPDATES)
    live_settings = MonitorSettings(
        host=DEFAULT_HOST,
        port=DEFAULT_PORT,
        client_id=DEFAULT_CLIENT_ID + 20,
        account=TARGET_ACCOUNT,
        market_data_type=MARKET_DATA_TYPES["Live"],
        quote_wait_seconds=2.0,
        refresh_seconds=5,
        auto_refresh=False,
        auto_reconnect=False,
        reconnect_backoff_seconds=10,
        wechat_webhook_url="",
        wechat_push_enabled=False,
        wechat_min_interval_seconds=300,
    )
    positions, all_positions = fetch_target_positions(ib, TARGET_ACCOUNT)
    tickers = subscribe_quotes_for_positions(ib, positions, live_settings)
    ib.sleep(live_settings.quote_wait_seconds)
    live_frame = positions_to_frame(positions, tickers, portfolio_items_by_key(ib, TARGET_ACCOUNT))
    live_summary = account_summary_frame(ib, TARGET_ACCOUNT)
    useful_cols = ["optionName", "localSymbol", "position", "bid", "ask", "mid", "last", "modelOptionPrice", "price", "priceSource", "marketValue", "valueSource", "unrealizedPnL", "estimatedUnrealizedPnL", "iv", "delta", "gamma", "theta", "vega", "missingData", "conId"]
    display(live_frame[[col for col in useful_cols if col in live_frame.columns]])
    display(greek_totals(live_frame))
    display(live_summary)
    ib.disconnect()
else:
    print("Live IB test skipped. Set RUN_LIVE_IB = True and TARGET_ACCOUNT first.")

Error 354, reqId 15: Requested market data is not subscribed. Check API status by selecting the Account menu then under Management choose Market Data Subscription Manager and/or availability of delayed data.Delayed market data is available.ZF SEP'26 (ZFU6) /TOP/ALL, contract: Future(conId=842590380, symbol='ZF', lastTradeDateOrContractMonth='20260930', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='ZFU6', tradingClass='ZF')
Error 354, reqId 72: Requested market data is not subscribed. Check API status by selecting the Account menu then under Management choose Market Data Subscription Manager and/or availability of delayed data.Delayed market data is available.ZF SEP'26 (ZFU6) /TOP/ALL, contract: Future(conId=842590380, symbol='ZF', lastTradeDateOrContractMonth='20260930', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='ZFU6', tradingClass='ZF')
Error 10090, reqId 3: Part of requested market data is not subscribed. Subscription-independent ticks are st

,optionName,localSymbol,position,bid,ask,mid,last,modelOptionPrice,price,priceSource,...,valueSource,unrealizedPnL,estimatedUnrealizedPnL,iv,delta,gamma,theta,vega,missingData,conId
0,ZF-20260615-106.75-P,VF3M6 P1067,-3.0,NaN,NaN,NaN,NaN,NaN,0.000001,portfolio,...,portfolio,112.02,112.0200,NaN,NaN,NaN,NaN,NaN,greeks unavailable,888539400
1,ZF-20260615-107-P,VF3M6 P1070,-1.0,NaN,NaN,NaN,NaN,1.988975e-02,0.012075,portfolio,...,portfolio,103.40,103.4000,0.042263,-1.910103e-01,1.451262,-1.988975e-02,0.011926,,888539208
2,ZF-20260615-107.25-C,VF3M6 C1072,-3.0,NaN,NaN,NaN,NaN,4.241283e-02,0.039062,portfolio,...,portfolio,-59.84,-59.8400,0.045638,3.301597e-01,1.863022,-4.241283e-02,0.014987,,888539242
3,ZF-20260615-107.5-C,VF3M6 C1075,-3.0,NaN,NaN,NaN,NaN,6.514466e-03,0.007515,portfolio,...,portfolio,27.00,27.0000,0.060627,6.528796e-02,0.560531,-6.514466e-03,0.005700,,888539362
4,ZF-20260616-105.5-P,GF3M6 P1055,1.0,NaN,NaN,NaN,NaN,4.739781e-08,0.000009,portfolio,...,portfolio,-9.52,-9.5200,0.052187,-6.812583e-07,0.000010,-4.739781e-08,0.000003,,888969065
5,ZF-20260616-105.75-P,GF3M6 P1057,1.0,NaN,NaN,NaN,NaN,2.097882e-07,0.000008,portfolio,...,portfolio,-17.34,-17.3400,0.047371,-3.155351e-06,0.000046,-2.097882e-07,0.000010,,888969068
6,ZF-20260616-106.5-P,GF3M6 P1065,-1.0,NaN,NaN,NaN,NaN,6.422049e-04,0.000003,portfolio,...,portfolio,76.41,76.4100,0.040119,-7.183729e-03,0.073626,-6.420598e-04,0.002645,,888969062
7,ZF-20260616-106.75-P,GF3M6 P1067,-2.0,NaN,NaN,NaN,NaN,1.126975e-02,0.007970,portfolio,...,portfolio,175.94,175.9400,0.041810,-8.216490e-02,0.507900,-1.073955e-02,0.012946,,888968718
8,ZF-20260616-107-P,GF3M6 P1070,-1.0,NaN,NaN,NaN,NaN,4.614795e-02,0.044050,portfolio,...,portfolio,94.86,94.8600,0.039899,-2.735286e-01,1.221048,-3.327588e-02,0.024434,,888968828
9,ZF-20260616-107.25-C,GF3M6 C1072,-2.0,NaN,NaN,NaN,NaN,7.383864e-02,0.077282,portfolio,...,portfolio,-40.81,-40.8100,0.040400,3.795842e-01,1.367822,-4.254238e-02,0.026501,,888968904


,method,deltaContracts,deltaMultiplier,gammaMultiplier,thetaMultiplier,vegaMultiplier
0,IB system ticker Greeks,0.335663,335.662605,-22977.722469,577.770948,-631.311835
1,Mid-price Greeks,NaN,NaN,NaN,NaN,NaN


,account,tag,value,currency
0,U16251798,AvailableFunds,-140.35,USD
1,U16251798,BuyingPower,11473.80,USD
2,U16251798,ExcessLiquidity,3013.66,USD
3,U16251798,FullAvailableFunds,-140.35,USD
4,U16251798,FullExcessLiquidity,3013.66,USD
5,U16251798,FullInitMarginReq,21074.03,USD
6,U16251798,FullMaintMarginReq,17935.39,USD
7,U16251798,GrossPositionValue,1921.91,USD
8,U16251798,InitMarginReq,21074.03,USD
9,U16251798,LookAheadAvailableFunds,-140.35,USD


In [3]:
import os
import requests

bark_url = "https://api.day.app/wF6yZEVtrVLqL7h2cnekwA/"

payload = {
    "title": "Bark 测试",
    "body": "推送已成功接入，可以在 Bark 中查看完整内容。",
    "group": "程序提醒",
    "isArchive": "1",
    "action": "alert"
}

response = requests.post(
    bark_url,
    json=payload,
    timeout=10
)

response.raise_for_status()
print(response.json())

{'code': 200, 'message': 'success', 'timestamp': 1781505723}


In [2]:
import threading
import time

from ibapi.client import EClient
from ibapi.wrapper import EWrapper


class IBNewsApp(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)

    def nextValidId(self, orderId: int):
        print("连接成功")
        self.reqNewsProviders()

    def newsProviders(self, news_providers):
        print("\n可用新闻源：")

        if not news_providers:
            print("没有返回任何 API 新闻源")
            self.disconnect()
            return

        self.news_provider_codes = []

        for provider in news_providers:
            code = getattr(
                provider,
                "code",
                getattr(provider, "providerCode", None),
            )
            name = getattr(
                provider,
                "name",
                getattr(provider, "providerName", None),
            )

            self.news_provider_codes.append(code)
            print(f"{code}: {name}")

        self.disconnect()

    def error(
        self,
        req_id,
        error_time,
        error_code,
        error_string,
        advanced_order_reject_json="",
    ):
        print(
            f"IBKR消息：reqId={req_id}, "
            f"time={error_time}, "
            f"code={error_code}, "
            f"message={error_string}"
        )


def run_loop(app):
    app.run()


app = IBNewsApp()

# TWS模拟账户常见端口：7497
# TWS实盘常见端口：7496
# 具体以TWS设置为准
app.connect(
    host="127.0.0.1",
    port=4001,
    clientId=88,
)

thread = threading.Thread(
    target=run_loop,
    args=(app,),
    daemon=True,
)
thread.start()

# time.sleep(10)

if app.isConnected():
    app.disconnect()

连接成功
IBKR消息：reqId=-1, time=1781508860705, code=2104, message=Market data farm connection is OK:hfarm
IBKR消息：reqId=-1, time=1781508860705, code=2107, message=HMDS data farm connection is inactive but should be available upon demand.apachmds
IBKR消息：reqId=-1, time=1781508860705, code=2158, message=Sec-def data farm connection is OK:secdefhk

可用新闻源：
BRFG: Briefing.com General Market Columns
BRFUPDN: Briefing.com Analyst Actions
DJ-N: Dow Jones Global Equity Trader
DJ-RTA: Dow Jones Top Stories Asia Pacific
DJ-RTE: Dow Jones Top Stories Europe
DJ-RTG: Dow Jones Top Stories Global
DJ-RTPRO: Dow Jones Top Stories Pro
DJNL: Dow Jones Newsletters


In [3]:
import threading
import time
from datetime import datetime

from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract


NEWS_PROVIDERS = (
    "BRFG+BRFUPDN+DJ-N+DJ-RTA+"
    "DJ-RTE+DJ-RTG+DJ-RTPRO+DJNL"
)


class IBStockNews(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)

        self.contract_found = False
        self.headlines = []
        self.finished = threading.Event()

    def nextValidId(self, order_id: int):
        print(f"连接成功，nextValidId={order_id}")

        contract = Contract()
        contract.symbol = "ORCL"
        contract.secType = "STK"
        contract.exchange = "SMART"
        contract.currency = "USD"
        contract.primaryExchange = "NYSE"

        self.reqContractDetails(
            reqId=1001,
            contract=contract,
        )

    def contractDetails(self, req_id, contract_details):
        if req_id != 1001 or self.contract_found:
            return

        self.contract_found = True

        contract = contract_details.contract
        con_id = contract.conId

        print(
            f"找到合约：{contract.symbol}, "
            f"conId={con_id}, "
            f"exchange={contract.primaryExchange}"
        )

        # 空白开始和结束时间表示查询最近新闻
        self.reqHistoricalNews(
            reqId=2001,
            conId=con_id,
            providerCodes=NEWS_PROVIDERS,
            startDateTime="",
            endDateTime="",
            totalResults=20,
            historicalNewsOptions=[],
        )

    def contractDetailsEnd(self, req_id):
        if req_id == 1001 and not self.contract_found:
            print("没有找到对应合约")
            self.finished.set()

    def historicalNews(
        self,
        req_id,
        news_time,
        provider_code,
        article_id,
        headline,
    ):
        item = {
            "time": news_time,
            "provider": provider_code,
            "article_id": article_id,
            "headline": headline,
        }

        self.headlines.append(item)

        number = len(self.headlines)

        print(
            f"\n[{number}] {news_time}"
            f"\n来源：{provider_code}"
            f"\n标题：{headline}"
            f"\n文章ID：{article_id}"
        )

    def historicalNewsEnd(self, req_id, has_more):
        print(
            f"\n新闻查询完成，共返回 "
            f"{len(self.headlines)} 条，hasMore={has_more}"
        )

        if not self.headlines:
            self.finished.set()
            return

        # 自动获取第一条新闻正文
        latest = self.headlines[0]

        print("\n正在请求第一条新闻的正文……")

        self.reqNewsArticle(
            reqId=3001,
            providerCode=latest["provider"],
            articleId=latest["article_id"],
            newsArticleOptions=[],
        )

    def newsArticle(
        self,
        req_id,
        article_type,
        article_text,
    ):
        print("\n" + "=" * 70)
        print("新闻正文")
        print("=" * 70)
        print(f"文章类型：{article_type}")
        print(article_text[:10000])

        self.finished.set()

    def error(
        self,
        req_id,
        error_time,
        error_code,
        error_string,
        advanced_order_reject_json="",
    ):
        # 忽略常见连接状态提示
        if error_code in {2104, 2106, 2107, 2158}:
            print(f"IBKR状态：{error_code} - {error_string}")
            return

        try:
            readable_time = datetime.fromtimestamp(
                error_time / 1000
            ).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            readable_time = str(error_time)

        print(
            f"IBKR错误：time={readable_time}, "
            f"reqId={req_id}, "
            f"code={error_code}, "
            f"message={error_string}"
        )


app = IBStockNews()

app.connect(
    host="127.0.0.1",
    port=4001,
    clientId=89,  # 不要与其他程序重复
)

api_thread = threading.Thread(
    target=app.run,
    daemon=True,
)
api_thread.start()

# 最多等待30秒
app.finished.wait(timeout=10)

if app.isConnected():
    app.disconnect()

连接成功，nextValidId=1
IBKR状态：2104 - Market data farm connection is OK:hfarm
IBKR状态：2107 - HMDS data farm connection is inactive but should be available upon demand.apachmds
IBKR状态：2158 - Sec-def data farm connection is OK:secdefhk
找到合约：ORCL, conId=272800, exchange=NYSE
IBKR状态：2106 - HMDS data farm connection is OK:ushmds

[1] 2026-06-13 01:31:00.0
来源：DJ-N
标题：{A:800015:L:en}Tech Trader: Oracle Has Been an AI Success. It May Not Be Enough for the Stock. -- Barron's
文章ID：DJ-N$1eaf099c

[2] 2026-06-12 20:04:00.0
来源：DJ-N
标题：{A:800015:L:en}Oracle Is an AI Success. It May Not Be Enough. -- Barrons.com
文章ID：DJ-N$1eae735b

[3] 2026-06-12 19:34:00.0
来源：DJ-N
标题：{A:800015:L:en}Oracle Is an AI Success. It May Not Be Enough. -- Barrons.com
文章ID：DJ-N$1eae67ed

[4] 2026-06-12 14:16:00.0
来源：DJ-N
标题：{A:800015:L:en}Stock Market Whipsaws; SpaceX IPO Rockets, Oracle Dives: Weekly Review -- IBD
文章ID：DJ-N$1eadbfda

[5] 2026-06-12 12:08:00.0
来源：DJ-N
标题：{A:800015:L:en}Dow Jones Futures Rise, Oil Skids On Iran Dea

In [6]:
import sys
from pathlib import Path

# 当前 Notebook 位于：
# IB-API/target_treasury_account_monitor/
ib_api_root = Path.cwd().parent

print("当前工作目录：", Path.cwd())
print("准备加入路径：", ib_api_root)

sys.path.insert(0, str(ib_api_root))

from news_api.ibkr_news_pipeline import IBKRNewsPipeline


PROVIDERS = (
    "BRFG+BRFUPDN+DJ-N+DJ-RTA+"
    "DJ-RTE+DJ-RTG+DJ-RTPRO+DJNL"
)

pipeline = IBKRNewsPipeline(
    symbol="ORCL",
    primary_exchange="NYSE",
    provider_codes=PROVIDERS,

    host="127.0.0.1",
    port=4001,              # 你的 IB Gateway 端口
    client_id=91,           # 避免与其他连接重复

    total_results=100,      # 最多拉取100条标题
    max_articles=30,        # 最多读取30篇正文
    local_timezone="Asia/Taipei",
    verbose=True,
)

df = pipeline.run_pipeline(timeout=180)

当前工作目录： e:\策略\IB-API\target_treasury_account_monitor
准备加入路径： e:\策略\IB-API
连接成功，nextValidId=1
IBKR状态：2104 - Market data farm connection is OK:hfarm
IBKR状态：2107 - HMDS data farm connection is inactive but should be available upon demand.apachmds
IBKR状态：2107 - HMDS data farm connection is inactive but should be available upon demand.ushmds
IBKR状态：2158 - Sec-def data farm connection is OK:secdefhk
找到合约：ORCL, conId=272800, primaryExchange=NYSE
IBKR状态：2106 - HMDS data farm connection is OK:ushmds
历史新闻返回 100 条；去重后 74 条；hasMore=True
正文完成 1/30: Tech Trader: Oracle Has Been an AI Success. It May Not Be Enough for the Stock.
正文完成 2/30: Oracle Is an AI Success. It May Not Be Enough.
正文完成 3/30: Stock Market Whipsaws; SpaceX IPO Rockets, Oracle Dives: Weekly Review
正文完成 4/30: Dow Jones Futures Rise, Oil Skids On Iran Deal Hopes; SpaceX IPO Indicated Highe
正文完成 5/30: Dow Jones Futures Rise, Oil Dives On Iran Deal Hopes; SpaceX IPO To Launch After
正文完成 6/30: Dow Jones Futures: SpaceX IPO To Launch Aft

In [12]:
df.iloc[3]

symbol                                                               ORCL
con_id                                                             272800
time_utc                                        2026-06-12T12:08:00+00:00
time_local                                      2026-06-12T20:08:00+08:00
provider                                                             DJ-N
publisher                                                             IBD
language                                                               en
article_id                                                  DJ-N$1ead7692
headline                Dow Jones Futures Rise, Oil Skids On Iran Deal...
duplicate_count                                                         4
article_type                                                          0.0
article_error                                                            
article_text            By Ed Carson\n\nDow Jones futures rose modestl...
headline_raw            {A:800015:L:en